# Parse logs
Stream text records into categorized Iceberg log tables.

In [ ]:
project_root = "."
source = "data/capture"
pattern = "*.log*"
recursive = True
timezone = None
start = None
end = None
fix_version = "4.4"
fix_dictionary = "data/fix"
null_values = ["", "null", "<null>", "n/a"]
rules = None
protocols = None
catalog = "rekep"
catalog_properties = {}
target_pattern = "logs.{category}"
instrument_source = "market.instruments"
instrument_snapshot_every = 3_600_000_000_000
static_values = {}
merge_by = True
batch_row_size = 65_536
commit_row_size = 250_000
limit = None

In [ ]:
import datetime
from pathlib import Path

import pyarrow
import pyarrow.compute as pc
import pyarrow.fs
from pyiceberg.expressions import And, GreaterThanOrEqual, LessThan, LessThanOrEqual

from rekep.filesystems import resolve
from rekep.fix.registry import FixRegistry
from rekep.fix.rules import Rules
from rekep.fix.transcribe import FixCodec
from rekep.iceberg import IcebergDataset
from rekep.market import Instrument
from rekep.market.event import DAY
from rekep.text import Log, LogRules, TextFile, TextFiles
from rekep.urls import Url


def _location(value):
    parsed = Url.from_string(str(value))
    if parsed.scheme in {"", "file", "local"} and not Path(parsed.path).is_absolute():
        parsed = Url.from_path(project_root).join(parsed.path)
    return parsed.into_string()


def _local_path(value):
    parsed = Url.from_string(str(value))
    if parsed.scheme not in {"", "file", "local"}:
        return str(value)
    path = Path(parsed.path)
    return str(path if path.is_absolute() else Path(project_root).resolve() / path)


def _unix_ns(value, *, upper=False):
    if value is None:
        return None
    text = str(value)
    date_only = len(text) == 10
    instant = datetime.datetime.fromisoformat(text.replace("Z", "+00:00"))
    if instant.tzinfo is None:
        instant = instant.replace(tzinfo=datetime.UTC)
    instant = instant.astimezone(datetime.UTC)
    if upper and date_only:
        instant += datetime.timedelta(days=1)
    epoch = datetime.datetime(1970, 1, 1, tzinfo=datetime.UTC)
    return (instant - epoch) // datetime.timedelta(microseconds=1) * 1_000


def _bounded(batch):
    lower, upper = _unix_ns(start), _unix_ns(end, upper=True)
    mask = None
    if lower is not None:
        mask = pc.greater_equal(batch.column("unix"), lower)
    if upper is not None:
        before = pc.less(batch.column("unix"), upper)
        mask = before if mask is None else pc.and_(mask, before)
    return batch if mask is None else batch.filter(mask)


def _window(lower, upper, column="unix"):
    predicates = []
    if lower is not None:
        predicates.append(GreaterThanOrEqual(column, lower))
    if upper is not None:
        predicates.append(LessThan(column, upper))
    return None if not predicates else predicates[0] if len(predicates) == 1 else And(*predicates)


event_rules = LogRules() if rules is None else LogRules.from_dict(rules)
protocol_rules = Rules() if protocols is None else Rules.from_dict(protocols)
registry = FixRegistry(cache_dir=_local_path(fix_dictionary), offline=True)
codec = FixCodec(
    rules=protocol_rules,
    registry=registry,
    fix_version=fix_version,
    null_values=frozenset(null_values),
)
declared = {
    "timezone": timezone,
    "static_values": static_values,
    "rules": event_rules,
    "codec": codec,
}
location = _location(source)
filesystem, path = resolve(location)
info = filesystem.get_file_info(path)
if info.type == pyarrow.fs.FileType.NotFound:
    raise FileNotFoundError(location)
rows = (
    TextFiles.from_folder(location, pattern=pattern, recursive=recursive, **declared)
    if info.type == pyarrow.fs.FileType.Directory
    else TextFile.from_url(location, **declared)
)
field = rows.into_struct_field()

In [ ]:
targets = {}
buffers = {}
held_rows = {}
read = written = skipped = 0
observed_lower = observed_upper = None


def _target(category):
    target = targets.get(category)
    if target is None:
        target = targets[category] = IcebergDataset(
            name=target_pattern.format(category=category),
            catalog=catalog,
            properties=dict(catalog_properties),
            struct=field,
            commit_row_size=commit_row_size,
            sort_by=("unix", "seq", "hash"),
        )
    return target


def _flush(category):
    global written, skipped
    batches = buffers.pop(category, [])
    count = held_rows.pop(category, 0)
    if not count:
        return
    target = _target(category)
    landed = target.append_arrow_table(pyarrow.Table.from_batches(batches), merge_by=merge_by)
    written += landed
    skipped += count - landed


for batch in rows.read_arrow_reader(batch_row_size=batch_row_size):
    batch = _bounded(batch)
    if limit is not None and read + batch.num_rows > limit:
        batch = batch.slice(0, max(0, limit - read))
    if not batch.num_rows:
        continue
    bounds = pc.min_max(batch.column("unix")).as_py()
    observed_lower = bounds["min"] if observed_lower is None else min(observed_lower, bounds["min"])
    observed_upper = (
        bounds["max"] + 1 if observed_upper is None else max(observed_upper, bounds["max"] + 1)
    )
    read += batch.num_rows
    categories = protocol_rules.into_arrow_category_array(
        batch.column("protocol"), batch.column("etype")
    )
    names = sorted(pc.unique(categories).to_pylist())
    for category in names:
        part = batch if len(names) == 1 else batch.filter(pc.equal(categories, category))
        buffers.setdefault(category, []).append(part)
        held_rows[category] = held_rows.get(category, 0) + part.num_rows
        if commit_row_size and held_rows[category] >= commit_row_size:
            _flush(category)
    if limit is not None and read >= limit:
        break
for category in list(buffers):
    _flush(category)

lower = _unix_ns(start) if start is not None else observed_lower
upper = _unix_ns(end, upper=True) if end is not None else observed_upper
instrument_table = IcebergDataset(
    name=instrument_source, catalog=catalog, properties=dict(catalog_properties)
)


def _instrument_seeds():
    if lower is None or not instrument_table.exists:
        return []
    recent = And(
        GreaterThanOrEqual("unix", lower - DAY),
        LessThanOrEqual("unix", lower),
    )
    reader = instrument_table.read_arrow_reader(
        Instrument.into_field(),
        row_filter=recent,
        order_by=("unix", "version", "hash"),
    )
    latest = {}
    for batch in reader:
        for row in batch.to_pylist():
            instrument = Instrument.from_dict(row)
            current = latest.get(instrument.xhash)
            if current is None or (instrument.unix, instrument.version, instrument.hash) > (
                current.unix,
                current.version,
                current.hash,
            ):
                latest[instrument.xhash] = instrument
    return sorted(latest.values(), key=lambda row: (row.unix, row.xhash))


def _market_logs():
    target = _target("market")
    if not target.exists:
        return
    reader = target.read_arrow_reader(
        Log.into_field(),
        row_filter=_window(lower, upper),
        order_by=("unix", "seq", "hash"),
    )
    for batch in reader:
        for row in batch.to_pylist():
            log = Log.from_dict(row)
            if not log.is_instrument_version:
                yield log


def _instrument_flush(held):
    if not held:
        return 0
    table = pyarrow.Table.from_pylist(
        [{**instrument.into_log().into_dict(), **static_values} for instrument in held],
        schema=field.into_arrow_schema(),
    )
    held.clear()
    return _target("market").append_arrow_table(table, merge_by=merge_by)


instrument_versions = instrument_written = 0
instrument_held = []
if lower is not None and upper is not None:
    versions = Instrument.from_logs(
        _market_logs(),
        registry=registry,
        fix_version=fix_version,
        instruments=_instrument_seeds(),
        snapshot_every=instrument_snapshot_every,
        snapshot_until=upper,
    )
    for instrument in versions:
        if instrument.unix < lower or instrument.unix >= upper:
            continue
        instrument_held.append(instrument)
        instrument_versions += 1
        if commit_row_size and len(instrument_held) >= commit_row_size:
            instrument_written += _instrument_flush(instrument_held)
    instrument_written += _instrument_flush(instrument_held)

result = {
    "read": read,
    "written": written + instrument_written,
    "skipped": skipped + instrument_versions - instrument_written,
    "raw_written": written,
    "instrument_versions": instrument_versions,
    "instrument_written": instrument_written,
    "targets": {category: target.name for category, target in targets.items()},
}
result